# RI #1: Query Rewriting, Corrective Retrieval, and Null Detection

After the retrieval and chunking comparisons in notebooks 02 and 03, this notebook asks what the answering policy earns.

Good-looking passages do not guarantee that a contract answers a question.
This notebook follows the decisions between retrieval and generation: rewriting
a query, accepting or retrying evidence, requiring citations, and abstaining.
It also isolates a common failure: answering a question about one contract
using a plausible clause from another.

By the end, you should be able to:

- Distinguish deterministic query cleanup from feedback-driven corrective rewriting.
- Read a saved retrieval–grade–action trace without treating its score as answer correctness.
- Evaluate citation requirements using identical raw model responses.
- Report null detection, answerable correctness, scope, and model usage on their proper denominators.

## 1. Experiment Overview

This notebook builds on [notebook 01](01_harness_and_baseline.ipynb).
Run the cells in order. Leave
`RUN_EXPENSIVE = False` for the first pass. This path reads the measured shared
cache and runs small local exercises; it needs no API key, downloads no model
weights, and makes no hosted calls. The output identifies the original measurement
separately from today's replay. A missing report is an error, not a substitute result.

Enabling the flag runs the **40-question policy comparison**, including model
grading, rewriting, generation, and separate correctness/grounding judging.
It is not a 40-call experiment. New chat calls use the shared generation, extraction, and judging roles through
the Vector proxy. Onboarding credentials or the `.env` fallback are resolved by
the shared helpers; embeddings remain local. Recorded results retain their original models.
Temperature zero removes sampling variance only and does not guarantee
identical outputs across serving backends. Replay preserves the measured sample.

In [1]:
import sys
from pathlib import Path

from IPython.display import Markdown
from IPython.display import display as render_display


# Replay is the first-run path. Fresh measurements require an explicit opt-in.
RUN_EXPENSIVE = False

REPO = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "aieng-advanced-rag").is_dir()),
    None,
)
if REPO is None:
    raise RuntimeError("Open this notebook from within the advanced-RAG checkout.")
RI = REPO / "implementations" / "regulatory_policy_qa"
sys.path.insert(0, str(REPO / "aieng-advanced-rag"))
sys.path.insert(0, str(RI))
from advanced_lab.common import saved  # noqa: E402 - use this source checkout


def show_table(headers, rows):
    """Display a compact table without requiring pandas or a model runtime."""

    def line(values):
        return "| " + " | ".join(str(value).replace("|", r"\|").replace("\n", " ") for value in values) + " |"

    render_display(Markdown("\n".join([line(headers), line(["---"] * len(headers)), *(line(row) for row in rows)])))


print("Execution mode:", "fresh experiment enabled" if RUN_EXPENSIVE else "saved reports + local exercises")

Execution mode: saved reports + local exercises


## 2. Replay a threshold intervention on real saved traces

Change `SUFFICIENT`, `INSUFFICIENT`, or `ATTEMPTS` and rerun this cell. Unlike
generating another answer, this intervention selects among attempts already
recorded in the 40-contract, 680-question harvest. It makes no model calls.
The five-passage reader budget stays fixed: replay cannot change what the
recorded generator read. Balanced accuracy scores answer-versus-abstain
decisions, not the semantic correctness of the resulting answers.

Always answer and always abstain both score 0.5 balanced decision accuracy.
The no-grader control reads the first generator decision. Keep these controls
when deciding whether another grade or retry helps. This is the earlier
40-contract experiment, not a full 123-contract measurement.

In [2]:
import json  # noqa: E402 - method exercise cell

from aieng.advanced_rag.evaluation.sweep import QueryTrace, control_rows, sweep  # noqa: E402 - method exercise cell
from aieng.advanced_rag.pipeline.grading import GradingThresholds  # noqa: E402 - method exercise cell


SUFFICIENT = 0.6
INSUFFICIENT = 0.0
ATTEMPTS = 2
traces = [QueryTrace.from_dict(row) for row in json.loads((RI / "outputs/traces_contract-nli_llm.json").read_text())]
threshold = GradingThresholds(sufficient=SUFFICIENT, insufficient=INSUFFICIENT)
replayed = sweep(traces, [threshold], attempts=(ATTEMPTS,), grader="llm", evidence_budget=5)
show_table(
    [
        "Policy",
        "Balanced decision accuracy",
        "Null false-answer rate",
        "Answerable refusal rate",
        "Logical model calls",
    ],
    [
        (
            row.label,
            f"{row.balanced_accuracy:.4f}",
            f"{row.null.false_answer_rate:.4f}",
            f"{row.answerable.abstain_rate:.4f}",
            row.model_calls,
        )
        for row in [*replayed, *control_rows(traces, evidence_budget=5)]
    ],
)
print("Recorded questions:", len(traces), "New model calls: 0")

| Policy | Balanced decision accuracy | Null false-answer rate | Answerable refusal rate | Logical model calls |
| --- | --- | --- | --- | --- |
| 0.60 / 0.00 | 0.7869 | 0.1553 | 0.2709 | 7173 |
| no grader | 0.7799 | 0.1553 | 0.2849 | 680 |
| always answer | 0.5000 | 1.0000 | 0.0000 | 680 |
| always abstain | 0.5000 | 0.0000 | 1.0000 | 0 |

Recorded questions: 680 New model calls: 0


## 3. Start with a Model-Free Rewrite

CUAD questions repeat a long instruction around the clause category.
`TemplateRewriter` strips that known boilerplate before retrieval. A
corrective `LLMRewriter` instead uses feedback from weak evidence to propose
another retrieval query, adding model calls and a chance to change meaning.

The examples below are invented mechanics examples. The CUAD template has
something to remove; an ordinary ContractNLI-style question may produce
`no-change`. An empty rewrite plan is not necessarily a failure. The final
answering task always remains the original question.

In [3]:
from aieng.advanced_rag.pipeline.rewrite import TemplateRewriter  # noqa: E402 - exercise cell


rewriter = TemplateRewriter()
rewrite_examples = [
    "Highlight the parts (if any) of this contract related to termination that should be reviewed by a lawyer.",
    "Must the receiving party delete every copy?",
]
for original in rewrite_examples:
    plan = rewriter.rewrite(original)
    print("Original:", original)
    print("Outcome:", plan.outcome, "Candidate queries:", plan.questions(include_original=False))

Original: Highlight the parts (if any) of this contract related to termination that should be reviewed by a lawyer.
Outcome: rewritten Candidate queries: ('termination',)
Original: Must the receiving party delete every copy?
Outcome: no-change Candidate queries: ()


## 4. Define the Controlled Policy Comparison

The first 20 ContractNLI contracts contain 340 questions. Before generation,
the experiment selects the first answerable and first null question per
contract: **40 questions, with 20 answerables and 20 nulls**. This balanced
demonstration is not the full slice or the full 123-contract experiment.

| Arm | Intervention | What remains fixed |
| --- | --- | --- |
| `citations-required` | Require a usable citation when parsing the response | Scoped dense evidence, original raw generation response. |
| `citations-optional` | Parse the **identical raw response** without the citation gate | Same question, evidence, model response; no second generation. |
| `dense-unscoped` | Search across the indexed contracts | Questions, embedding model, generator, five-passage reader cap. |
| `corrective-loop` | Add LLM grading, feedback rewriting, and up to two retrieval attempts | Original questions, contract scope, model, reader cap. |

The citation pair isolates a parser requirement. The corrective arm changes
several stages together, so its difference is a **policy-level comparison**,
not an isolated estimate of the benefit of rewriting. The unscoped arm
deliberately violates the closed-contract task boundary as a diagnostic control.

In [4]:
if RUN_EXPENSIVE:
    from advanced_lab.answering import measure_policies

    report = measure_policies(documents=20)
else:
    report = saved("policies")

print("Report:", "fresh measurement" if RUN_EXPENSIVE else "saved measurement; zero new model calls")
print("Originally measured:", report["generated_at"])

Report: saved measurement; zero new model calls
Originally measured: 2026-10-20T19:44:22.831725+00:00


## 5. Keep Decision Quality, Correctness, and Grounding Separate

| Measurement | Population | Meaning |
| --- | --- | --- |
| Abstention recall | 20 nulls | Fraction of null questions refused. |
| False-answer rate | 20 nulls | Fraction of null questions on which the system attempts an answer. |
| Abstention precision | All refused questions | Fraction of refusals whose gold label is null. |
| Answerable correctness | All 20 answerables, including refusals | Agreement with the annotated reference, judged with `judge_answer`. |
| Grounding | Returned evidence | Whether the answer is supported by the evidence it was given. |

Correctness and grounding use separate judge passes. Correctness sees the
reference without retrieved text; grounding sees the final evidence without
the reference. A canonical abstention is incorrect on an answerable question
and vacuously grounded. Thus a high grounding rate can coexist with an
unhelpful system, or with answers about the wrong contract. Generation and
judging share a model, so these are automated annotation-agreement measurements.

In [5]:
policy_rows = []
for name, arm in report["arms"].items():
    correct = sum(row["judgement"]["correct"] for row in arm["outcomes"] if not row["is_null"])
    null_metrics = arm["null_detection"]
    policy_rows.append(
        (
            name,
            f"{correct}/{arm['answerable']}",
            arm["answerable_abstentions"],
            f"{null_metrics['abstain_recall']:.4f}",
            f"{null_metrics['false_answer_rate']:.4f}",
        )
    )
show_table(
    ["Arm", "Correct / 20 answerables", "Answerable abstentions", "Null recall", "Null false-answer rate"], policy_rows
)
required = {row["query_id"]: row for row in report["arms"]["citations-required"]["outcomes"]}
optional = {row["query_id"]: row for row in report["arms"]["citations-optional"]["outcomes"]}
print(
    "Citation pair shares every raw response:",
    all(required[key]["raw_response_sha256"] == optional[key]["raw_response_sha256"] for key in required),
)

| Arm | Correct / 20 answerables | Answerable abstentions | Null recall | Null false-answer rate |
| --- | --- | --- | --- | --- |
| citations-required | 15/20 | 4 | 1.0000 | 0.0000 |
| citations-optional | 16/20 | 3 | 1.0000 | 0.0000 |
| dense-unscoped | 9/20 | 10 | 0.0000 | 1.0000 |
| corrective-loop | 15/20 | 4 | 1.0000 | 0.0000 |

Citation pair shares every raw response: True


The saved citation pair scores **15/20** correct with required citations and
**16/20** without the gate. Because both parse identical raw responses, the
difference comes from citation validation rather than a new generation sample.
One answer is not a general argument for removing citations; a resolved
citation is also not proof that the answer is correct.

The corrective arm scores **15/20**. The unscoped arm falsely answers **all
20 nulls**, showing why matching some retrieved text is insufficient for the
contract-specific task. Use the source audit below to inspect the mechanism.

## 6. Trace a Real Corrective Retry

A retry follows the path `retrieve → grade → choose an action → rewrite if
needed → retrieve again → answer or abstain`. The trace records the actual
retrieval query, grade, action, and evidence IDs. A larger grade means the
grader preferred those passages; it does not establish reference correctness.
The cell selects a saved multi-attempt example when one exists.

In [6]:
loop_rows = report["arms"]["corrective-loop"]["outcomes"]
example = next((row for row in loop_rows if len(row["attempt_trace"]) > 1), loop_rows[0])
print("Question ID:", example["query_id"], "Final abstention:", example["abstained"])
for attempt in example["attempt_trace"]:
    print(
        "Attempt",
        attempt["attempt"],
        "Strategy:",
        attempt["strategy"],
        "Grade:",
        attempt["score"],
        "Action:",
        attempt["action"],
    )
    print("Retrieval query:", attempt["question"])
    print("Source IDs:", attempt["evidence_ids"])

Question ID: 1::nda-11 Final abstention: True
Attempt 1 Strategy: original Grade: 0.0 Action: rewrite
Retrieval query: Receiving Party shall not reverse engineer any objects which embody Disclosing Party's Confidential Information.
Source IDs: ['1::0007', '1::0004', '1::0006', '1::0008', '1::0005', '1::0009', '1::0014', '1::0011', '1::0003', '1::0002']
Attempt 2 Strategy: llm Grade: 0.0 Action: answer
Retrieval query: Receiving Party shall not reverse engineer any objects which embody Disclosing Party's Confidential Information
Source IDs: ['1::0007', '1::0004', '1::0006', '1::0008', '1::0005', '1::0009', '1::0014', '1::0011', '1::0002', '1::0012']


Compare the first and second retrieval queries. Did the rewrite retain the
original obligation or add a restriction? Did different source IDs appear?
Explain the final abstention decision using the trace before consulting the
answer judge. This trace was reconstructed from cached responses, not new calls.

## 7. Price the Loop Without Treating Cache Hits as Free Design

`responses_requested` counts calls plus cache hits in the recorded execution.
It describes how many model completions the policy asks for, while `calls`
records new paid requests in that execution. A replay can have zero calls
while still representing a costly policy. Judging is a separate evaluation
cost and is excluded from the generation/loop rows below.

In [7]:
show_table(
    ["Recorded phase", "Responses requested", "New calls in that execution", "Cache hits"],
    [
        (phase, row["responses_requested"], row["calls"], row["cache_hits"])
        for phase, row in report["generation_and_loop_requests"].items()
    ],
)
print("Saved report execution:", report["execution"])
print(
    "Recorded total, including judging:", {key: report["usage"][key] for key in ("calls", "total_tokens", "cache_hits")}
)

| Recorded phase | Responses requested | New calls in that execution | Cache hits |
| --- | --- | --- | --- |
| generation:citation-pair | 40 | 0 | 40 |
| generation:dense-unscoped | 40 | 0 | 40 |
| pipeline:corrective-loop | 434 | 0 | 434 |

Saved report execution: cached model responses only
Recorded total, including judging: {'calls': 0, 'total_tokens': 0, 'cache_hits': 836}


The saved sample requests **434 model responses** for the corrective policy
versus **40 single-pass generation responses**, without improving its 15/20
answerable correctness. These are logical response requests, including cache
hits; they are not a new invoice incurred by running this notebook.

The larger historical threshold experiment found only **0.008 balanced-accuracy
gain (paired McNemar p = 0.26; 95% paired bootstrap interval −0.004 to 0.021)** for about **9.2 times the calls**. That measurement,
documented in [THRESHOLDS.md](THRESHOLDS.md), supports keeping the recommended
participant pipeline single-pass. The loop remains an opt-in experiment. The
smaller demonstration illustrates behavior and does not replace that study.

## 8. Audit the Retrieval Scope Directly

In the shared retrieval interface, `scope=query.doc_id` restricts the search
to that contract; `scope=None` searches the indexed corpus. An empty document
string is a filter, not a synonym for a global search. The independent local
audit below uses the same 40 selected questions and records actual source IDs.
Retrieval failure and MRR use only its 20 answerables.

In [8]:
if RUN_EXPENSIVE:
    from advanced_lab.scope import measure_scope

    scope_report = measure_scope()
else:
    scope_report = saved("scope")

show_table(
    ["Retrieval scope", "Questions with foreign hits / 40", "Failure @10 on 20 answerables", "MRR @10"],
    [
        (
            name,
            arm["queries_with_cross_contract_hits"],
            f"{arm['retrieval']['retrieval_failure']:.4f}",
            f"{arm['retrieval']['mrr']:.4f}",
        )
        for name, arm in scope_report["arms"].items()
    ],
)
scope_example = next(row for row in scope_report["arms"]["unscoped"]["outcomes"] if row["cross_contract_hits"])
print("Example question ID:", scope_example["query_id"])
print("Required contract:", scope_example["expected_doc_id"])
print("Actually retrieved contracts:", sorted(set(scope_example["retrieved_doc_ids"])))

| Retrieval scope | Questions with foreign hits / 40 | Failure @10 on 20 answerables | MRR @10 |
| --- | --- | --- | --- |
| scoped | 0 | 0.0000 | 0.7200 |
| unscoped | 40 | 0.7500 | 0.1133 |

Example question ID: 1::nda-11
Required contract: 1
Actually retrieved contracts: ['1', '2', '21', '22', '24', '25', '26', '36', '38']


Every unscoped sample query retrieves at least one foreign-contract passage;
the scoped control has none. This directly verifies that the scope
intervention occurred. It also explains how an answer can be well grounded
in returned evidence and still be wrong for the requested contract.

## 9. A retry control that separates rewriting from another draw

Remote main's later `judge_not_stated_retry.py` investigation adds an essential
control: send the identical prompt again after a refusal, without rewriting or
retrieving. The frozen report below has 680 questions, uses the original 35B
generator, and uses `Qwen3_8-Flash-Next` for judging. It is a separate experiment
from the 40-question citation/scope study above.

The rewrite retry and resampling have no detectable difference in the recorded
paired comparison. The report also retains regenerated decisions because some
temperature-zero responses did not reproduce the harvest's original refusal.
Do not mix trace-decision scores with regenerated-decision scores. A new draw
can explain a retry gain; it is not evidence that rewriting improved retrieval.

In [9]:
retry = json.loads((RI / "outputs/not_stated_retry.json").read_text())
resample = retry["explore_resample_retry"]
show_table(
    ["Control comparison", "Paired McNemar p"],
    [
        ("Resample versus generator-only", resample["correctness_scored"]["paired_vs_generator_only"]["mcnemar_p"]),
        ("Resample versus rewrite retry", resample["correctness_scored"]["paired_vs_rewrite_retry"]["mcnemar_p"]),
    ],
)
print(retry["finding"])
print("These are saved judgments, not new generations.")

| Control comparison | Paired McNemar p |
| --- | --- |
| Resample versus generator-only | 0.0357 |
| Resample versus rewrite retry | 1.0 |

Scored by judged correctness, the retry on the generator's not-stated answer beats generator-only, and a retry that re-sends the identical first-attempt prompt on a fresh call matches it question for question at fewer calls. The rewrite and the second retrieval add nothing measurable here: the gain is a second draw from a generator whose abstain-or-answer decision changes on about one answerable question in ten when the same prompt is sent again at temperature zero. The harvest cached responses on the exact request, so inside the traces an identical retry prompt always returned the first refusal, which hid this. Generation on every attempt uses the original question, so the rewrite changes only which chunks reach the generator and in what order.
These are saved judgments, not new generations.


For the public-domain QA and RAGAS demonstration continue with
[notebook 06](06_regulatory_qa_and_ragas.ipynb). The [CUAD comparison](outputs/cuad_null/COMPARISON.md)
rechecks the GraphRAG null claim on five fixed contracts; it is a bounded pilot,
not the full CUAD test set. The 123-contract answer/null measurement remains
deferred: notebook 03's full-split retrieval study does not complete it.

## 10. Design a Follow-Up and State Its Limits

- **Rewrite only:** compare original and rewritten retrieval on identical
  questions, scope, and qrels. Inspect meaning changes before comparing metrics.
- **Policy only:** add a retry while keeping the model and reader budget fixed.
  Save original outcomes, per-attempt sources, correctness, and usage by stage.
- **Citation gate only:** preserve identical raw responses as in this notebook;
  generating two independent answers would add sampling and serving variability.
- **Evaluation:** include answerable refusals in correctness and all nulls in
  false-answer rate. Use held-out questions and document-aware uncertainty
  before making a general recommendation.

The **123-contract ContractNLI experiment remains cache-only** under the current
project scope. [Its coverage record](outputs/full_contract_nli/cache_status.json)
documents missing entries; this 40-question study supplies no full-split score.
The [CUAD null-arm follow-up](outputs/cuad_null/COMPARISON.md) is a separate
five-contract GraphRAG comparison, with its own population and limitations.

**Check your understanding:** Why is always abstaining inadequate? Why can
source-grounded answers fail the task? Which comparison isolates citation
validation, and which changes several policy components together?

Continue with [notebook 05](99_starter_agent.ipynb). See [PIPELINE.md](PIPELINE.md)
for the stage interfaces, [GRAPHRAG_NULL_ARM.md](GRAPHRAG_NULL_ARM.md) for the
larger null experiment, and [ANSWERING_EXPERIMENTS.md](ANSWERING_EXPERIMENTS.md)
for fresh-run controls and costs.